# 2 · Entrenamiento y evaluación

> **Tipo de ML:** `{{ ml_type }}`{% if ml_type == 'redes_neuronales' %}  · Arquitectura: `{{ nn_model }}`{% endif %}{% if ml_type in ['supervisado', 'hibrido'] and model_type != 'todos' %}  · Modelo activo: `{{ model_type }}`{% endif %}


## 0. Entorno


In [ ]:
# Verificar que el entorno está activo
import sys
print(f'Python: {sys.version}')
{% if ml_type == 'redes_neuronales' %}
import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
{% endif %}
{% if use_xgboost or model_type == 'XGBoost' %}
import xgboost; print(f'XGBoost: {xgboost.__version__}')
{% endif %}
{% if use_lightgbm or model_type == 'LightGBM' %}
import lightgbm; print(f'LightGBM: {lightgbm.__version__}')
{% endif %}

## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from {{ project_slug }}.utils.paths import (
    PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR, REPORTS_DIR,
)

{% if ml_type == 'supervisado' %}
from {{ project_slug }}.models.train_model import train_models, load_models
from {{ project_slug }}.models.predict_model import evaluate_models, predict_new, DECISION_THRESHOLD
from {{ project_slug }}.visualization.visualize import plot_feature_importance
{% elif ml_type == 'no_supervisado' %}
from {{ project_slug }}.models.train_model import train_models, find_optimal_k
from {{ project_slug }}.models.predict_model import evaluate_models
from {{ project_slug }}.visualization.visualize import (
    plot_elbow_and_silhouette, plot_dendrogram, plot_pca_variance,
)
{% elif ml_type == 'redes_neuronales' %}
import torch
from {{ project_slug }}.models.train_model import (
    train_models, load_model, MODEL_NAME,
)
from {{ project_slug }}.models.predict_model import evaluate_models, predict_new
{% elif ml_type == 'hibrido' %}
from {{ project_slug }}.models.train_model import train_models, load_models
from {{ project_slug }}.models.predict_model import evaluate_models
from {{ project_slug }}.visualization.visualize import plot_feature_importance
{% endif %}

## 2. Cargar datos procesados


In [ ]:
{% if ml_type in ['supervisado', 'redes_neuronales', 'hibrido'] %}
X_train = pd.read_csv(PROCESSED_DATA_DIR / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DATA_DIR / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DATA_DIR / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED_DATA_DIR / 'y_test.csv').squeeze()
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Clases: {sorted(y_train.unique().tolist())}')
print(f'Balance (train): {y_train.value_counts(normalize=True).round(3).to_dict()}')
{% elif ml_type == 'no_supervisado' %}
X = pd.read_csv(PROCESSED_DATA_DIR / 'X_processed.csv').values
print(f'Shape: {X.shape}')
{% endif %}

## 3. {% if ml_type == 'no_supervisado' %}Selección del número de clusters{% else %}Configuración{% endif %}

{% if ml_type == 'no_supervisado' %}
Tres métodos complementarios para elegir k:
- **Dendrograma**: línea vertical más larga sin cruzar ninguna horizontal → cuenta las ramas.
- **Método del codo**: busca el punto donde la inercia deja de bajar bruscamente.
- **Silhouette Score**: maximizar (rango [-1, 1]; más alto es mejor).
{% elif ml_type == 'redes_neuronales' %}
Parámetros del entrenamiento. Ajusta según tu hardware y dataset.
{% elif ml_type == 'supervisado' %}
Modelo activo: `{{ model_type }}`. Cambia `model_type` en json y regenera para entrenar otro.
{% endif %}


In [ ]:
{% if ml_type == 'no_supervisado' %}
# Dendrograma
plot_dendrogram(X, method='ward')

# Varianza PCA (útil si hay muchas dimensiones)
plot_pca_variance(X)

# Elbow + Silhouette
metrics = find_optimal_k(X, k_range=range(2, 11))
plot_elbow_and_silhouette(metrics)
{% elif ml_type == 'redes_neuronales' %}
EPOCHS         = 50
BATCH_SIZE     = 32
LR             = 1e-3
CHECKPOINT     = 10    # guardar checkpoint cada N épocas (0 = desactivado)
INPUT_DIM      = X_train.shape[1]
OUTPUT_DIM     = y_train.nunique()
print(f'Arquitectura : {{ nn_model }}')
print(f'input_dim    : {INPUT_DIM}')
print(f'output_dim   : {OUTPUT_DIM}')
print(f'device       : {"cuda" if torch.cuda.is_available() else "cpu"}')
{% elif ml_type == 'supervisado' %}
# Configuración: model_type activo = '{{ model_type }}'
# tune_knn=True  → busca el k óptimo para KNN (lento en datasets grandes)
# cv_evaluate=True → muestra F1 5-fold de cada modelo antes de guardar
TUNE_KNN    = True
CV_EVALUATE = True
{% elif ml_type == 'hibrido' %}
# Estrategias disponibles: pca_clf | umap_clf | kmeans_features | iso_feature | semi_supervisado
STRATEGY = 'pca_clf'
{% endif %}

## 4. Entrenar


In [ ]:
{% if ml_type == 'supervisado' %}
models = train_models(X_train, y_train, tune_knn=TUNE_KNN, cv_evaluate=CV_EVALUATE)
{% elif ml_type == 'no_supervisado' %}
N_CLUSTERS = 3   # ajusta tras ver el dendrograma y el método del codo
models = train_models(X, n_clusters=N_CLUSTERS)
{% elif ml_type == 'redes_neuronales' %}
models = train_models(
    X_train, y_train,
    input_dim=INPUT_DIM, output_dim=OUTPUT_DIM,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    lr=LR, checkpoint_every=CHECKPOINT,
)
print(f'Modelo entrenado: {MODEL_NAME}')
{% elif ml_type == 'hibrido' %}
models = train_models(X_train, y_train, strategy=STRATEGY,
                      tune_knn=True, cv_evaluate=True)
{% endif %}

## 5. Evaluar


In [ ]:
{% if ml_type == 'supervisado' %}
threshold  = DECISION_THRESHOLD   # ajustar si clases muy desbalanceadas
df_results = evaluate_models(
    models, X_train, y_train, X_test, y_test, threshold=threshold
)
df_results.sort_values('F1_test', ascending=False)
{% elif ml_type == 'no_supervisado' %}
evaluate_models(models, X)
{% elif ml_type == 'redes_neuronales' %}
df_results = evaluate_models(
    models, X_test, y_test,
    num_classes=OUTPUT_DIM,
)
df_results   # métricas + CSVs guardados en reports/
{% elif ml_type == 'hibrido' %}
df_results = evaluate_models(
    models, X_train, y_train, X_test, y_test
)
df_results.sort_values('F1_test', ascending=False)
{% endif %}

## 6. Importancia de variables


In [ ]:
{% if ml_type in ['supervisado', 'hibrido'] %}
feature_names = X_train.columns.tolist()
plot_feature_importance(models, feature_names)
from IPython.display import Image
Image(FIGURES_DIR / 'feature_importance.png')
{% elif ml_type == 'redes_neuronales' %}
# Importancia de features vía permutation importance (model-agnostic)
from sklearn.inspection import permutation_importance
import torch

class _SklearnWrapper:
    """Envuelve el modelo PyTorch para sklearn.inspection."""
    def __init__(self, model, device):
        self.model = model
        self.device = device
    def predict(self, X):
        self.model.eval()
        with torch.no_grad():
            t = torch.tensor(X, dtype=torch.float32).to(self.device)
            return torch.argmax(self.model(t), dim=1).cpu().numpy()
    def score(self, X, y):
        from sklearn.metrics import accuracy_score
        return accuracy_score(y, self.predict(X))

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
wrapper = _SklearnWrapper(models[MODEL_NAME].to(device), device)
perm    = permutation_importance(
    wrapper, X_test.values, y_test.values,
    n_repeats=10, random_state=42, scoring='accuracy',
)
imp_df = pd.DataFrame({
    'feature':    X_train.columns,
    'importance': perm.importances_mean,
    'std':        perm.importances_std,
}).sort_values('importance', ascending=False)
imp_df.plot.bar(x='feature', y='importance', yerr='std', figsize=(12, 4))
plt.title(f'Permutation Importance — {{ nn_model }}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_importance_{{ nn_model }}.png', dpi=150)
plt.show()
imp_df
{% else %}
pass  # No disponible para este tipo de ML
{% endif %}

## 7. Predicción sobre nuevos datos


In [ ]:
{% if ml_type == 'supervisado' %}
best_name  = df_results.sort_values('F1_test', ascending=False).iloc[0]['Modelo']
best_model = models[best_name]
print(f'Mejor modelo: {best_name}')

# Predecir sobre nuevos datos:
# from {{ project_slug }}.features.build_features import process_input
# X_new   = process_input(df_nuevos)
# preds, probs = predict_new(best_model, X_new, threshold=threshold)
{% elif ml_type == 'no_supervisado' %}
# KMeans tiene .predict(); AgglomerativeClustering no.
# model = joblib.load(MODELS_DIR / 'KMeans.joblib')
# labels = model.predict(X_new)
{% elif ml_type == 'redes_neuronales' %}
# Cargar pesos guardados (útil si el kernel se reinició)
# model = load_model(input_dim=INPUT_DIM, output_dim=OUTPUT_DIM)

# Predecir sobre nuevos datos y exportar CSV:
# X_new  = pd.read_csv('...')
# preds  = predict_new(
#     model, X_new, num_classes=OUTPUT_DIM,
#     export_csv=True, out_name='predicciones_nuevas',
# )
# print(preds)
{% elif ml_type == 'hibrido' %}
best_name  = df_results.sort_values('F1_test', ascending=False).iloc[0]['Modelo']
best_model = models[best_name]
print(f'Mejor modelo: {best_name}')

# preds, probs = predict_new(best_model, X_new)
{% endif %}

## 8. Profiling (opcional)

Si el entrenamiento es lento, identifica dónde se va el tiempo:

```bash
make profile
snakeviz reports/profile.prof
```
